<a href="https://colab.research.google.com/github/bOOgeyMan-p/Assignment/blob/main/HMM_PROJECT_PIYUSH_KUMAR_25124034_BEC_108.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                                  [0.0, 0.9, 0.1, 0.0, 0.0],
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]])
emission_nuc_codes = {'A': 0,
                      'C': 1,
                      'G': 2,
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00],
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [2]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [3]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


-43.89740030179307

In [4]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [5]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)

-43.944833355027704


In [6]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)

-42.58225552052512


In [7]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [8]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [9]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)

-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides.

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .]
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .]
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [13]:
import numpy as np
import math

# Initiate two matrices:
# viterbi_value_matrix: to store the values described in the documentation above
# viterbi_trace_matrix: to store the path the lead to the the maximum value in each cell
# For example, the first column of viterbi_trace_matrix will be
# [0] indicating start state released `C`: even though not possible - but we just initiate
# [1] indicating Exon state released `C`:
# [2] indicating 5'ss state released `C`: even though not possible - but we just initiate
# [3] indicating Intron state released `C`:
# [4] indicating end state released `C`: even though not possible - but we just initiate

# Number of hidden states (including 's' and 'e' for Viterbi calculation)
num_states = len(states)
# Length of the observed sequence
seq_len = len(query_sequence)

# Initialize Viterbi value matrix with -infinity for log probabilities
viterbi_value_matrix = np.full((num_states, seq_len), -np.inf)
# Initialize Viterbi trace matrix to store backpointers (previous state index)
viterbi_trace_matrix = np.zeros((num_states, seq_len), dtype=int)

# Initialization for the first observation (query_sequence[0])
first_observation_code = emission_nuc_codes[query_sequence[0]]
start_state_idx = states['s'] # Index of the 'start' pseudo-state

# Iterate through all possible hidden states
for state_idx in range(num_states):
    # The probability of reaching state_idx at the first observation
    # is P(state_idx | 's') * P(observation[0] | state_idx)
    # In log scale: log(P(state_idx | 's')) + log(P(observation[0] | state_idx))

    # Get transition probability from 's' to current state_idx
    trans_prob_from_s = state_transition_prob[start_state_idx][state_idx]

    # Get emission probability for the first observation from current state_idx
    emission_prob_first_obs = emission_probs[state_idx][first_observation_code]

    # Calculate log probability, handling log(0) which results in -np.inf
    if trans_prob_from_s > 0 and emission_prob_first_obs > 0:
        viterbi_value_matrix[state_idx, 0] = math.log(trans_prob_from_s) + math.log(emission_prob_first_obs)
    else:
        viterbi_value_matrix[state_idx, 0] = -np.inf # Log of 0 is -infinity

    # For the first column, the trace back is to the 's' (start) state, which has index 0
    viterbi_trace_matrix[state_idx, 0] = start_state_idx

print("Viterbi Value Matrix (initialized first column):")
print(viterbi_value_matrix)
print("\nViterbi Trace Matrix (initialized first column):")
print(viterbi_trace_matrix)

Viterbi Value Matrix (initialized first column):
[[       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [-1.38629436        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf]
 [       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         

### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND**

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [14]:
def calculate_prob_for_a_node(col_idx, current_state_idx, query_sequence, viterbi_value_matrix, state_transition_prob, emission_probs, emission_nuc_codes):
    current_nuc_code = emission_nuc_codes[query_sequence[col_idx]]
    max_log_prob = -np.inf
    best_prev_state_idx = -1

    # Iterate over all possible previous states
    for prev_state_idx in range(num_states):
        # Log probability of transition from prev_state to current_state
        trans_log_prob = -np.inf
        if state_transition_prob[prev_state_idx][current_state_idx] > 0:
            trans_log_prob = math.log(state_transition_prob[prev_state_idx][current_state_idx])

        # Viterbi value of the previous state
        prev_viterbi_value = viterbi_value_matrix[prev_state_idx, col_idx - 1]

        # If previous viterbi value is -inf, this path is not possible
        if prev_viterbi_value == -np.inf:
            continue

        # Current path's log probability
        current_path_log_prob = prev_viterbi_value + trans_log_prob

        # Update max_log_prob and best_prev_state_idx
        if current_path_log_prob > max_log_prob:
            max_log_prob = current_path_log_prob
            best_prev_state_idx = prev_state_idx

    # Add emission probability for the current observation at the current state
    emission_log_prob = -np.inf
    if emission_probs[current_state_idx][current_nuc_code] > 0:
        emission_log_prob = math.log(emission_probs[current_state_idx][current_nuc_code])

    # Final Viterbi value for the current cell
    final_log_prob = max_log_prob + emission_log_prob if max_log_prob != -np.inf else -np.inf

    return final_log_prob, best_prev_state_idx

# Fill the Viterbi matrices for the remaining observations
for col_idx in range(1, seq_len):
    for current_state_idx in range(num_states):
        # The start state 's' (index 0) and end state 'e' (index 4) do not emit observations
        # or transition to themselves in a meaningful way for internal steps
        # For this problem, we are specifically interested in states E, 5, I.
        # 's' is only for initial transition, 'e' for final transition.
        # We should only calculate for states that can actually be reached and emit.
        # In the context of the given problem, 's' and 'e' states often don't emit
        # and '5' (splice site) might have specific transition rules.
        # Let's consider only E, 5, I for the inner loop.
        # The problem statement's hint implies filling all rows with some value,
        # but based on emission_probs, only E, 5, I emit.
        # Let's adhere to the structure of the provided emission_probs where s and e have 0 emission prob.
        if id2state[current_state_idx] == 's' or id2state[current_state_idx] == 'e':
            viterbi_value_matrix[current_state_idx, col_idx] = -np.inf
            viterbi_trace_matrix[current_state_idx, col_idx] = -1 # No valid predecessor
            continue

        val, trace = calculate_prob_for_a_node(col_idx, current_state_idx, query_sequence, viterbi_value_matrix, state_transition_prob, emission_probs, emission_nuc_codes)
        viterbi_value_matrix[current_state_idx, col_idx] = val
        viterbi_trace_matrix[current_state_idx, col_idx] = trace

print("\nCompleted Viterbi Value Matrix:")
print(viterbi_value_matrix)
print("\nCompleted Viterbi Trace Matrix:")
print(viterbi_trace_matrix)



Completed Viterbi Value Matrix:
[[        -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf]
 [ -1.38629436  -2.87794924  -4.36960411  -5.86125899  -7.35291387
   -8.84456875 -10.33622362 -11.8278785  -13.31953338 -14.81118825
  -16.30284313 -17.79449801 -19.28615288 -20.77780776 -22.26946264
  -23.76111751 -25.25277239 -26.74442727 -28.23608214 -29.72773702
  -31.2193919  -32.71104677 -34.20270165 -35.69435653 -37.1860114
  -38.67766628]
 [        -inf         -inf         -inf         -inf -11.15957636
          -inf -11.19844713         -inf -14.18175689 -18.61785074
  -20.10950562 -21.6011605  -20.14837639         -inf -26.07612513
  -24.62334102 -29.05943488         -inf -29.09830565         -in

In [15]:
# Write a function to trace the state path that gave the maximum probability.
# This will be the final result.


# HINT: You should first find the maximum value in the last column of `viterbi_value_matrix`,
# because that is the one with the largest probability.
# The index of that value is the state of the last nucleotide.

def backtrack_viterbi_path(viterbi_value_matrix, viterbi_trace_matrix, id2state, query_sequence):
    seq_len = len(query_sequence)
    num_states = viterbi_value_matrix.shape[0]

    # Find the state with the maximum probability in the last column
    last_column = viterbi_value_matrix[:, seq_len - 1]
    max_prob_last_state = -np.inf
    last_state_idx = -1

    # Iterate over all states to find the maximum in the last column
    # Exclude 's' (index 0) and 'e' (index 4) if they are not meant to be end states for the path
    # Based on the problem (e.g. gene finding), 'e' might be the final state to consider for probability.
    # Let's consider all states that could lead to the end, then specifically choose the end state if applicable.
    # For this problem, 'e' is typically the absorbing end state.
    # However, the Viterbi algorithm typically ends by finding the max in the last column
    # over all possible internal states, and then transitioning to the 'e' state if one exists.
    # If 'e' is a valid state to be in at the last observation, we'd look for max in its row.
    # Given `emission_probs[4]` is all zeros, state 'e' itself cannot emit.
    # So, we should look for the max *before* the 'e' state transition.
    # The problem implies we want the path for the actual query sequence length.
    # So we look at the last column, which corresponds to the last nucleotide.
    # The last state of the optimal path for the observation query_sequence[seq_len-1]
    # cannot be 's' or 'e' as they do not emit.

    # Find the state (E, 5, or I) with the maximum probability at the last observation point
    for state_idx in range(num_states):
        if id2state[state_idx] != 's' and id2state[state_idx] != 'e': # Only consider states E, 5, I
            if viterbi_value_matrix[state_idx, seq_len - 1] > max_prob_last_state:
                max_prob_last_state = viterbi_value_matrix[state_idx, seq_len - 1]
                last_state_idx = state_idx

    # If no valid path was found (e.g., all were -np.inf), handle gracefully
    if last_state_idx == -1:
        return "No valid path found", -np.inf

    # Backtrack to find the full path
    best_path = [last_state_idx]
    current_trace_idx = last_state_idx
    for col in range(seq_len - 1, 0, -1):
        prev_state_idx = viterbi_trace_matrix[current_trace_idx, col]
        best_path.append(prev_state_idx)
        current_trace_idx = prev_state_idx

    # The path is built in reverse, so reverse it
    best_path.reverse()

    # Convert state indices to state names
    best_path_names = [id2state[idx] for idx in best_path]

    # The first element in `best_path` will be the state that emitted the first observation.
    # The problem setup has an implicit 's' state that transitions to the first actual state.
    # We might want to remove the initial 's' state from the final output if it's implicitly handled.
    # The viterbi_trace_matrix[state_idx, 0] is initialized to 's' (0), so this 's' is likely part of it.
    # Let's clean it up to represent the path of actual emitting states.

    # If the first state in best_path_names is 's', remove it as it's a pseudo-state
    if best_path_names[0] == 's':
        best_path_names = best_path_names[1:]

    return "".join(best_path_names), max_prob_last_state

# Get the most probable path and its log probability
most_probable_path, final_log_prob = backtrack_viterbi_path(viterbi_value_matrix, viterbi_trace_matrix, id2state, query_sequence)

print("\nMost Probable Hidden State Sequence:", most_probable_path)
print("Final Log Probability:", final_log_prob)



Most Probable Hidden State Sequence: EEEEEEEEEEEEEEEEEEEEEEEEEE
Final Log Probability: -38.677666280562796
